# Initialize `oil_network` — top-level master orchestrator

Runs every stage of the project pipeline in the correct dependency order. Toggle individual stages on/off via the flags in the next cell — useful for partial re-runs (e.g. re-load EIA data without rebuilding the asset graph, or re-bind variable_assignments without re-fetching EIA).

## Stages

| # | Stage | Sub-orchestrator | Drops/recreates | Typical time |
|---|---|---|---|---|
| 1 | Asset graph + variables (skeleton) | [initialize_oil_logistics_network.ipynb](initialize_oil_logistics_network.ipynb) | `oil_network` schema | ~30 s |
| 2 | Typed metadata (production assets) | [initialize_oil_network_metadata.ipynb](initialize_oil_network_metadata.ipynb) | `oil_network_metadata` schema | ~5 s |
| 3 | Source-data loaders (EIA, future: CER, AER…) | [initialize_oil_network_data_loader.ipynb](initialize_oil_network_data_loader.ipynb) | `oil_network_data_loader` schema | **~5–10 min** (network-bound) |
| 4 | Assignments (timeseries catalogue + facts + variable_assignments) | [initialize_oil_network_assignments.ipynb](initialize_oil_network_assignments.ipynb) | UPSERT only | ~10 s |

## Dependencies

- Stage 4 needs **both** Stage 1 (variables to bind to) **and** Stage 3 (raw EIA data to read from).
- Stage 2 is independent — it only depends on Stage 1's `oil_network.assets` rows.
- Stage 3 is fully independent of Stages 1+2 (writes only to its own `oil_network_data_loader` schema).
- If you toggle Stage 4 ON without re-running Stage 3, the existing EIA data is reused. If you toggle Stage 1 OFF and Stage 4 ON, make sure the `oil_network` schema already exists from a prior run.

## Stage flags

Set each flag to `True` (run) or `False` (skip).

In [1]:
RUN_ASSET_GRAPH      = True    # Stage 1: build oil_network schema + load asset_graph.json
RUN_METADATA         = True    # Stage 2: build oil_network_metadata + load typed metadata
RUN_DATA_LOADER      = True    # Stage 3: pull EIA data into oil_network_data_loader  (~5-10 min)
RUN_ASSIGNMENTS      = True    # Stage 4: write timeseries + timeseries_data + variable_assignments

## Run

In [ ]:
import os
import subprocess
import sys
from datetime import datetime
from pathlib import Path

# Load .env so EIA_API_KEY + PG_* propagate to child notebook processes via
# inherited os.environ. We walk UP from cwd to find .env because VS Code runs
# notebooks with the workspace root as cwd (not the notebook's directory), so
# a hard-coded relative path can miss it and cause SILENT FAILURES — Stage 3
# (EIA loader) returns success with zero data, Stage 4 returns success with
# zero resolver output, and the pipeline reports green while producing nothing.
try:
    from dotenv import load_dotenv
    _env_loaded = None
    for _base in (Path.cwd(), *Path.cwd().parents):
        _candidate = _base / ".env"
        if _candidate.exists():
            load_dotenv(_candidate)
            _env_loaded = _candidate
            break
    if _env_loaded:
        print(f"loaded .env from {_env_loaded}")
    else:
        print("WARNING: no .env found in cwd or any parent — relying on shell environment")
except ImportError:
    pass  # python-dotenv not installed; env vars must come from the shell

# HARD CHECK: abort BEFORE running stages if EIA_API_KEY is missing. Without it,
# Stage 3 (EIA loader) silently no-ops and the pipeline reports green while
# producing zero rows downstream. We surface that failure mode here, not at
# verify_state.py time.
if not os.environ.get("EIA_API_KEY"):
    raise RuntimeError(
        "EIA_API_KEY is not set. Add it to Stage1/.env (or export it in your "
        "shell) before running this notebook. Without it, Stage 3 will silently "
        "produce zero rows and the whole pipeline will report success with an "
        "empty DB."
    )

STAGES = [
    ("1. Asset graph",       RUN_ASSET_GRAPH,  "initialize_oil_logistics_network.ipynb",      600),
    ("2. Metadata",          RUN_METADATA,     "initialize_oil_network_metadata.ipynb",       600),
    ("3. Source-data loaders", RUN_DATA_LOADER, "initialize_oil_network_data_loader.ipynb",   1800),
    ("4. Assignments",       RUN_ASSIGNMENTS,  "initialize_oil_network_assignments.ipynb",    600),
]


def stamp():
    return f"[{datetime.now():%H:%M:%S}]"


def run_notebook(nb_path, timeout_sec):
    cmd = [
        sys.executable, "-m", "jupyter", "nbconvert",
        "--to", "notebook",
        "--execute",
        "--inplace",
        f"--ExecutePreprocessor.timeout={timeout_sec}",
        nb_path,
    ]
    return subprocess.run(cmd, capture_output=True, text=True)


results = []
started = datetime.now()

for label, flag, nb, timeout in STAGES:
    if not flag:
        print(f"{stamp()} ⏭  {label} — SKIPPED (flag = False)")
        results.append((label, "skipped", None))
        continue
    if not Path(nb).exists():
        print(f"{stamp()} ✗  {label} — FILE MISSING ({nb})")
        results.append((label, "missing", nb))
        continue
    t0 = datetime.now()
    print(f"{stamp()} ▶  {label}: running {nb} (timeout {timeout}s)...")
    res = run_notebook(nb, timeout)
    dt = (datetime.now() - t0).total_seconds()
    if res.returncode != 0:
        tail = res.stderr.strip().splitlines()[-15:]
        print(f"{stamp()} ✗  {label} — FAILED (exit {res.returncode}, {dt:.0f}s)")
        results.append((label, "failed", "\n".join(tail)))
    else:
        print(f"{stamp()} ✓  {label} — done ({dt:.0f}s)")
        results.append((label, "ok", f"{dt:.0f}s"))

elapsed = (datetime.now() - started).total_seconds()
print()
print("=" * 80)
print(f"PIPELINE SUMMARY  (total elapsed {elapsed:.0f}s)")
print("=" * 80)
for label, status, info in results:
    icon = {"ok":"✓", "skipped":"⏭", "missing":"✗", "failed":"✗"}[status]
    suffix = f"  ({info})" if info else ""
    print(f"  {icon} {label:<35} {status}{suffix}")

failed = [r for r in results if r[1] == "failed"]
if failed:
    print()
    print("=" * 80)
    print("FAILURES — last 15 lines of each child notebook's stderr")
    print("=" * 80)
    for label, _, err in failed:
        print()
        print(f"--- {label} ---")
        print(err)